In [1]:
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

def download_and_plot_case(case_name, date_str, bbox):
    """
    Downloads, loads, and plots NLDAS, MRMS-FLASH, and LSR datasets for a specific case.
    bbox format: [min_lon, max_lon, min_lat, max_lat]
    """
    print(f"Starting processing for {case_name} on {date_str}...")
    
    # ---------------------------------------------------------
    # WORKFLOW STEP 1 & 2: Define Case, Download & Load Data
    # ---------------------------------------------------------
    # Note: Replace the placeholder paths below with the actual 
    # data fetching commands (e.g., requests, boto3, or pydap) 
    # used in your specific environment to pull from NCEI/Iowa Mesonet.
    
    nldas_file = f"data/nldas_{date_str}.nc"
    mrms_file = f"data/mrms_flash_{date_str}.nc"
    lsr_file = f"data/lsr_{date_str}.geojson"
    
    # Ensure directories exist
    os.makedirs("data", exist_ok=True)
    os.makedirs("output", exist_ok=True)
    
    # Mocking the load process. In a live environment, these will load the downloaded files.
    try:
        nldas_ds = xr.open_dataset(nldas_file)
        mrms_ds = xr.open_dataset(mrms_file)
        gdf_lsr = gpd.read_file(lsr_file)
        
        # Spatial Subsetting based on the bounding box
        nldas_crop = nldas_ds.sel(lon=slice(bbox[0], bbox[1]), lat=slice(bbox[2], bbox[3]))
        mrms_crop = mrms_ds.sel(lon=slice(bbox[0], bbox[1]), lat=slice(bbox[2], bbox[3]))
    except FileNotFoundError:
        print(f"Data files for {case_name} not found. Skipping plotting.")
        return

    # ---------------------------------------------------------
    # PLOTTING LOGIC
    # ---------------------------------------------------------
    fig = plt.figure(figsize=(10, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Plot NLDAS Soil Moisture
    nldas_crop.plot(
        ax=ax, cmap="YlGnBu", vmin=0, vmax=0.45,
        cbar_kwargs={"label": "NLDAS Soil Moisture"}
    )
    
    # Plot MRMS Max Soil Saturation
    mrms_crop.where(mrms_crop > 0).plot(
        ax=ax, cmap="OrRd", alpha=0.7, vmin=0, vmax=100,
        cbar_kwargs={"label": "MRMS Max Soil Saturation (%)"}
    )
    
    # Plot Local Storm Reports
    if not gdf_lsr.empty:
        gdf_lsr.plot(
            ax=ax, color='red', marker='^', markersize=50, 
            edgecolor='black', label='Floods'
        )
        plt.legend(loc='lower right')
    
    # Map Formatting
    ax.add_feature(cfeature.STATES)
    ax.set_extent(bbox) 
    plt.title(f"{case_name} - {date_str}")
    
    # Save Image
    output_filename = f"output/{case_name.replace(' ', '_').lower()}_{date_str}.png"
    plt.savefig(output_filename, bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"Successfully generated map: {output_filename}")

if __name__ == "__main__":
    # Test run for a single case
    download_and_plot_case("Iowa Floods", "2024-06-20", [-97.5, -89.5, 40.0, 44.0])

Starting processing for Iowa Floods on 2024-06-20...
Data files for Iowa Floods not found. Skipping plotting.


In [2]:
import multiprocessing
from process_case import download_and_plot_case

# Define the cases from the table
CASES = [
    {"name": "Iowa Floods", "date": "2024-06-20", "bbox": [-97.5, -89.5, 40.0, 44.0]},
    {"name": "Hurricane Harvey", "date": "2017-08-27", "bbox": [-98.0, -93.0, 27.0, 31.0]},
    {"name": "Hurricane Ida", "date": "2021-08-29", "bbox": [-92.0, -88.0, 28.0, 31.0]},
    {"name": "Eastern KY Floods", "date": "2022-07-28", "bbox": [-84.5, -81.5, 36.5, 38.5]},
    {"name": "CA Atmospheric River", "date": "2023-01-09", "bbox": [-124.0, -118.0, 34.0, 40.0]},
    {"name": "Vermont Floods", "date": "2023-07-10", "bbox": [-73.5, -71.5, 42.5, 45.0]}
]

def worker(case_dict):
    """Wrapper function to unpack dictionary for the target processing function."""
    download_and_plot_case(
        case_name=case_dict["name"],
        date_str=case_dict["date"],
        bbox=case_dict["bbox"]
    )

if __name__ == "__main__":
    print(f"Initializing batch processing for {len(CASES)} cases...")
    
    # Set up a multiprocessing pool. 
    # Adjust 'processes' based on your machine's CPU cores.
    num_cores = multiprocessing.cpu_count()
    
    with multiprocessing.Pool(processes=min(num_cores, len(CASES))) as pool:
        pool.map(worker, CASES)
        
    print("All cases have been processed successfully.")

ModuleNotFoundError: No module named 'process_case'